# STEP10. DMD annotation → 프레임 단위 눈 상태 GT

## 분석 질문

- DMD 의 drowsiness annotation(ASAM OpenLABEL)을 프레임 단위 눈 상태 정답표로 변환할 수 있는가, 그리고 그 프레임 번호가 mosaic 영상 프레임과 1:1로 대응하는가.
- 관련 가설: 해당 없음 (데이터 구축 단계)

## 입력

| 구분 | 경로 |
|---|---|
| DMD annotation JSON 16개 | `build_dmd_eye_dataset.dmd_dir()` — `config.DATA_DIR` 하위에서 해석 |
| DMD mosaic 영상 16개 | 같음 |
| 이전 STEP 결과 | 없음 |

## 출력

| 파일 | 위치 |
|---|---|
| `<video>_frame_gt.csv` (16개) | `config.OUTPUTS_DIR / "dmd_gt"` |
| `_summary.csv` | `config.OUTPUTS_DIR / "dmd_gt"` |
| `_alignment.csv` | `config.OUTPUTS_DIR / "dmd_gt"` |

경로는 직접 쓰지 않고 `config` 변수로부터 조립한다 (STYLE_GUIDE 4·17절).

## 전체 수행 흐름

**PART A — annotation 구조 확인**

1. 설정·경로
2. 한 영상 파싱 (스키마·라벨 종류)
3. mosaic ↔ annotation 프레임 정렬 검증

**PART B — 16개 배치 변환**

4. GT CSV · 요약 · 정렬 리포트 저장
5. 프레임 라벨 구성과 평가 제외 비율

## PART A — annotation 구조 확인

### 목적

- annotation 이 어떤 단위(프레임 / 구간)로 어떤 라벨을 담고 있는지 실데이터로 확인한다.
- 눈 상태 라벨을 이진 GT(Closed / Open)로 바꿀 때 무엇을 버려야 하는지 정한다.

In [10]:
# [셀 1] 설정 · 경로 (import·경로·시드는 여기서 한 번만)

# --- 저장소 루트 부트스트랩 (모든 노트북 공통, 수정 금지) ---
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}")
# --- 부트스트랩 끝 ---

# 프로젝트 모듈은 전부 src/ 에 평탄하게 둔다. 아직 패키지가 아니므로 sys.path 로 붙인다.
_src = str(config.PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

# config.setup_korean_font() 는 이 프로젝트 config.py 에 없다. 그림 라벨은 영문으로 쓴다.

import pandas as pd
import dmd_annotation as D
import build_dmd_eye_dataset as B

# 데이터셋 위치는 B.data_subdir 로 해석한다. data/raw/<name> 과 data/<name> 두
# 레이아웃을 모두 받아들여, 폴더를 옮겨도 노트북마다 경로를 고치지 않는다.
DMD_DIR = B.dmd_dir()
GT_DIR = config.OUTPUTS_DIR / "dmd_gt"
GT_DIR.mkdir(parents=True, exist_ok=True)

JSON_SUFFIX = "_rgb_ann_drowsiness.json"
AVI_SUFFIX = "_rgb_mosaic.avi"

jsons = sorted(DMD_DIR.glob(f"*{JSON_SUFFIX}"))
if not jsons:
    raise FileNotFoundError(f"annotation JSON 을 찾지 못했습니다: {config._rel(DMD_DIR)}")

print("annotation JSON :", len(jsons), "개")
print("출력 폴더       :", config._rel(GT_DIR))

annotation JSON : 16 개
출력 폴더       : outputs\dmd_gt


### 결정 박스 1 — 전이 프레임(opening / closing)을 어떻게 처리할 것인가

- 문제: `eyes_state` 는 open / close / opening / closing / undefined 5종이다. Eye CNN 은 Closed / Open 2종만 출력한다. 전이 프레임을 Closed 로 볼 것인가, Open 으로 볼 것인가, 평가에서 제외할 것인가.
- 선택: **제외한다.** `eye_gt_binary` 를 두어 close=1, open=0, 그 외=−1(제외)로 표기한다.
- 근거: opening / closing 은 눈꺼풀이 움직이는 중간 상태여서 정답이 애매하다. 어느 쪽으로 붙여도 그 선택이 Closed-Recall 을 직접 흔든다. 대안(전이를 Closed 에 포함)은 Recall 을 인위적으로 올리므로 쓰지 않는다. 제외 비율은 PART B 5단계에서 수치로 보고하고, 그 비율이 크면 결과 해석에 반영한다.
- 사전 계획이다(탐색적 선택 아님).

In [11]:
# [셀 2] 한 영상 파싱 — 스키마와 라벨 종류 확인
import json
from collections import Counter

jp = str(jsons[0])
raw = json.load(open(jp, encoding="utf-8"))["openlabel"]

meta = D.video_meta(raw)
action_types = Counter(a["type"] for a in raw["actions"].values())
streams = {k: (v["stream_properties"].get("total_frames"),
               v["stream_properties"].get("sync", {}).get("frame_shift"))
           for k, v in raw["streams"].items()}

print("영상 :", Path(jp).name.replace(JSON_SUFFIX, ""))
print("스키마 :", raw["metadata"].get("schema_version"))
print("action 타입 :", dict(action_types))
print("stream (total_frames, frame_shift) :", streams)
print("메타 :", {k: meta[k] for k in
               ["ann_frames", "face_total_frames", "face_frame_shift",
                "gender", "age", "glasses", "setup", "weather"]})

df = pd.DataFrame(D.build_frame_table(raw))
print("\n프레임 표 shape :", df.shape)
print("eye_state 분포 :", df["eye_state"].value_counts().to_dict())
print("eye_gt_binary  :", df["eye_gt_binary"].value_counts().to_dict())
df.head(6)

영상 : gA_1_s5_2019-03-14T14;26;17+01;00
스키마 : 1.0.0
action 타입 : {'eyes_state/open': 1, 'eyes_state/close': 1, 'eyes_state/opening': 1, 'eyes_state/closing': 1, 'blinks/blinking': 1, 'yawning/Yawning with hand': 1, 'yawning/Yawning without hand': 1}
stream (total_frames, frame_shift) : {'face_camera': (5480, 0), 'body_camera': (5427, 54), 'hands_camera': (5407, 74)}
메타 : {'ann_frames': 5481, 'face_total_frames': 5480, 'face_frame_shift': 0, 'gender': 'Male', 'age': 47, 'glasses': True, 'setup': 'Car Stopped', 'weather': 'Rainy'}

프레임 표 shape : (5481, 7)
eye_state 분포 : {'open': 3596, 'opening': 778, 'close': 660, 'closing': 446, 'none': 1}
eye_gt_binary  : {0: 3596, -1: 1225, 1: 660}


,frame,eye_state,eye_closed,eye_gt_binary,is_blink,is_yawn,yawn_type
0,0,open,0,0,0,0,none
1,1,open,0,0,0,0,none
2,2,open,0,0,0,0,none
3,3,open,0,0,0,0,none
4,4,open,0,0,0,0,none
5,5,open,0,0,0,0,none


### 관찰 결과

- 라벨은 프레임이 아니라 **프레임 구간(action)** 단위로 들어 있고, `eyes_state` / `blinks` / `yawning` 3계열이다.
- `streams` 는 face / body / hands 3개이고 face 만 `frame_shift = 0` 이다. body(+54) · hands(+74) 는 시점이 밀려 있어 face 타임라인만 써야 한다.
- 명시적 `drowsy` 또는 `microsleep` 라벨은 없다. 졸음은 파생 지표(긴 close = microsleep 후보, close 비율 = PERCLOS)로만 만들 수 있다.
- `eye_gt_binary` 는 close / open / 제외 3값으로 나뉜다.

### 목적

- annotation 프레임 번호를 mosaic 영상 프레임 번호로 그대로 써도 되는지 확인한다.
- `face_camera.total_frames` 와 `frame_intervals` 가 1 차이 나는 영상이 있어, 어느 쪽이 실제 프레임 수인지 확정한다.

In [12]:
# [셀 3] mosaic ↔ annotation 정렬 검증 (한 영상)
avi = Path(jp.replace(JSON_SUFFIX, AVI_SUFFIX))
mfc = D.mosaic_frame_count(str(avi)) if avi.exists() else None

print("annotation frames      :", meta["ann_frames"])          # frame_end + 1
print("face_camera total_frames:", meta["face_total_frames"])  # 스트림 메타데이터
print("mosaic frames (실측)    :", mfc)
print("정렬 :", mfc == meta["ann_frames"])

# face_total 이 1 작은 영상에서 마지막 프레임이 미라벨인지 확인한다.
# 미라벨이면 eye_gt_binary = -1 로 이미 제외되므로 정렬을 보정할 필요가 없다.
tail = df.tail(1)[["frame", "eye_state", "eye_gt_binary"]]
print("\n마지막 프레임 :", tail.to_dict("records"))

annotation frames      : 5481
face_camera total_frames: 5480
mosaic frames (실측)    : 5481
정렬 : True

마지막 프레임 : [{'frame': 5480, 'eye_state': 'none', 'eye_gt_binary': -1}]


## PART B — 16개 배치 변환

### 목적

- 16개 영상 전부를 프레임 GT CSV 로 변환하고, 영상·subject 메타와 정렬 결과를 한 표로 남긴다.
- 이 CSV 가 STEP11(눈 crop 생성)과 STEP13(프레임 단위 평가)의 유일한 정답 소스가 된다.

In [13]:
# [셀 4] 전체 배치 — GT CSV + 요약 + 정렬 리포트
def longest_run(flags) -> int:
    """연속 1의 최대 길이. microsleep 후보 구간 길이를 재기 위한 값."""
    best = cur = 0
    for v in flags:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best


def count_events(flags) -> int:
    """0 -> 1 전환 횟수. 깜빡임을 프레임 수가 아니라 '횟수'로 세기 위함."""
    prev, n = 0, 0
    for v in flags:
        if v and not prev:
            n += 1
        prev = v
    return n


# 라벨 구성 집계도 이 루프에서 함께 모은다. 셀 5 가 JSON 16개를 다시 파싱하면
# 같은 계산을 두 번 하게 된다.
summary, align = [], []
comp, binc = Counter(), Counter()

for jp_i in map(str, jsons):
    base = Path(jp_i).name.replace(JSON_SUFFIX, "")
    ol = D.load_openlabel(jp_i)
    m = D.video_meta(ol)
    rows = D.build_frame_table(ol)
    pd.DataFrame(rows).to_csv(GT_DIR / f"{base}_frame_gt.csv", index=False)

    for r in rows:
        comp[r["eye_state"]] += 1
        binc[r["eye_gt_binary"]] += 1

    closed = [r["eye_closed"] for r in rows]
    blinks = [r["is_blink"] for r in rows]
    n = len(rows)

    avi_i = Path(jp_i.replace(JSON_SUFFIX, AVI_SUFFIX))
    mfc_i = D.mosaic_frame_count(str(avi_i)) if avi_i.exists() else None

    summary.append(dict(
        video=base, subject=base.split("_s5")[0], gender=m["gender"], age=m["age"],
        glasses=m["glasses"], setup=m["setup"], weather=m["weather"],
        ann_frames=m["ann_frames"], closed_frames=sum(closed),
        closed_pct=round(sum(closed) / n * 100, 1), max_closed_run=longest_run(closed),
        yawn_frames=sum(r["is_yawn"] for r in rows), blink_events=count_events(blinks)))
    align.append(dict(
        video=base, ann_frames=m["ann_frames"], face_total=m["face_total_frames"],
        face_frame_shift=m["face_frame_shift"], mosaic_frames=mfc_i,
        aligned=(mfc_i == m["ann_frames"]) if mfc_i else None))

sum_df = pd.DataFrame(summary)
al_df = pd.DataFrame(align)
sum_df.to_csv(GT_DIR / "_summary.csv", index=False)
al_df.to_csv(GT_DIR / "_alignment.csv", index=False)

print("저장 :", config._rel(GT_DIR))
print(f"정렬 성공 : {int(al_df['aligned'].sum())} / {len(al_df)}")
print(f"face_total 이 ann_frames 보다 1 작은 영상 : "
      f"{int((al_df.ann_frames - al_df.face_total == 1).sum())} 개")
sum_df[["video", "subject", "gender", "glasses", "setup", "ann_frames",
        "closed_frames", "closed_pct", "max_closed_run"]]

저장 : outputs\dmd_gt
정렬 성공 : 16 / 16
face_total 이 ann_frames 보다 1 작은 영상 : 4 개


,video,subject,gender,glasses,setup,ann_frames,closed_frames,closed_pct,max_closed_run
0,gA_1_s5_2019-03-14T14;26;17+01;00,gA_1,Male,True,Car Stopped,5481,660,12.0,60
1,gA_5_s5_2019-03-13T09;06;49+01;00,gA_5,Male,False,Car Stopped,5287,550,10.4,54
2,gB_10_s5_2019-03-12T10;35;20+01;00,gB_10,Male,False,Car Stopped,6055,1050,17.3,121
3,gB_10_s5_2019-03-13T14;17;28+01;00,gB_10,Male,False,Car Stopped,5417,614,11.3,76
4,gB_6_s5_2019-03-13T13;37;11+01;00,gB_6,Male,False,Car Stopped,5410,176,3.3,39
5,gB_7_s5_2019-03-13T13;55;52+01;00,gB_7,Male,False,Car Stopped,5405,616,11.4,98
6,gB_9_s5_2019-03-07T16;31;48+01;00,gB_9,Male,False,Car Stopped,5292,646,12.2,88
7,gC_13_s5_2019-03-12T10;03;00+01;00,gC_13,Female,False,Car Stopped,5380,165,3.1,42
8,gC_14_s5_2019-03-12T09;18;58+01;00,gC_14,Male,False,Car Stopped,5576,477,8.6,115
9,gE_29_s5_2019-03-15T13;51;09+01;00,gE_29,Female,True,Car Stopped,5363,953,17.8,81


> 이 표는 영상별 subject 메타(성별·안경·촬영 환경)와 눈 감김 통계를 보여준다. `setup` 이 전부 동일하고 `closed_pct` 가 영상마다 크게 다르다는 두 가지가 이후 split 설계를 좌우한다.
> GT 소스: DMD 공식 OpenLABEL annotation. 정렬 판정 기준: mosaic 실측 프레임 수 == `frame_intervals.frame_end + 1`.

In [14]:
# [셀 5] 프레임 라벨 구성과 평가 제외 비율 (셀 4 에서 누적한 집계를 그대로 쓴다)
N = sum(comp.values())
usable = binc[0] + binc[1]
print("16개 프레임 구성(%) :", {k: round(v / N * 100, 1) for k, v in comp.items()})
print(f"\n총 프레임        : {N:,}")
print(f"이진 GT 사용 가능 : {usable:,} ({usable/N*100:.1f}%)")
print(f"  close (=1)     : {binc[1]:,} ({binc[1]/N*100:.1f}%)")
print(f"  open  (=0)     : {binc[0]:,} ({binc[0]/N*100:.1f}%)")
print(f"제외 (=-1)       : {binc[-1]:,} ({binc[-1]/N*100:.1f}%)")
print(f"\n사용 가능 프레임 내 Closed 비율 : {binc[1]/usable*100:.1f}%"
      f"  (Closed:Open = 1 : {binc[0]/binc[1]:.1f})")
print("안경 착용 subject :",
      sorted(sum_df.loc[sum_df.glasses, "subject"].unique().tolist()))

16개 프레임 구성(%) : {'open': 64.0, 'closing': 11.4, 'opening': 14.3, 'close': 10.0, 'none': 0.0, 'undefined': 0.2}

총 프레임        : 88,226
이진 GT 사용 가능 : 65,302 (74.0%)
  close (=1)     : 8,827 (10.0%)
  open  (=0)     : 56,475 (64.0%)
제외 (=-1)       : 22,924 (26.0%)

사용 가능 프레임 내 Closed 비율 : 13.5%  (Closed:Open = 1 : 6.4)
안경 착용 subject : ['gA_1', 'gE_29', 'gZ_36']


### 관찰 결과

- 전이·미라벨 프레임이 전체의 약 4분의 1을 차지해 이진 평가에서 빠진다.
- 사용 가능 프레임 안에서 Closed 비율이 한 자릿수 후반대이고, Open 이 Closed 의 6배 이상이다.
- `setup` 은 16개 영상 모두 `Car Stopped` 로 동일하다.
- 안경 착용 subject 는 13명 중 3명이다.
- `face_camera.total_frames` 가 `ann_frames` 보다 1 작은 영상이 있으나, 그 마지막 프레임은 `eye_state = none` 으로 이미 제외 대상이다.

## 해석

- annotation 은 프레임 단위 눈 상태 GT 로 바로 쓸 수 있고, mosaic 프레임 번호를 그대로 키로 사용해도 된다. 별도 오프셋 보정이 필요하지 않다.
- Closed 가 Open 의 6배 이상 적다는 것은, DMD 를 학습 데이터로 그냥 합치면 모델이 Open 으로 치우쳐 Closed-Recall 이 떨어진다는 뜻이다. 표본 수를 클래스별로 조정할 근거가 여기서 나온다. 조정 설계는 STEP11 에서 다룬다.
- 안경 착용 subject 가 3명뿐이라는 제약은 split 설계에 직접 영향을 준다. 안경은 눈 검출·감김 판정의 난이도를 크게 높이는 요인이므로, 3명을 train 과 test 에 어떻게 배분하는지가 일반화 측정의 신뢰도를 좌우한다.
- `setup` 이 전부 `Car Stopped` 이므로 이 데이터로 검증되는 것은 "연출된 눈 감김을 검출하는 능력"이며, "실제 주행 중 졸음을 예측하는 능력"이 아니다.

## 한계

이 STEP 의 결과는 눈 상태 라벨의 정확도를 보증하지 않으며, 졸음 자체의 정답으로 확대 해석하지 않는다.

- **연출 상황**: 16개 영상 모두 `setup = Car Stopped` 다. 실제 주행 중 졸음이 아니라 정지 상태에서 연기한 눈 감김·하품이다.
- **졸음 라벨 없음**: `drowsy` / `microsleep` 라벨이 annotation 에 존재하지 않는다. 졸음 관련 지표는 모두 파생값이다.
- **전이 프레임 손실**: 전체의 약 4분의 1이 이진 평가에서 빠진다. 눈이 감기는 순간을 놓치는지는 이 GT 로 측정할 수 없다.
- **클래스 불균형**: Closed 가 Open 의 6분의 1 이하다. 정확도(Accuracy)는 이 데이터에서 의미가 약하다.
- **표본 규모**: 13명 / 16영상이다. subject 단위 결론의 신뢰구간이 넓다.
- **안경 표본**: 안경 착용 subject 3명. 안경 조건의 성능 차이를 통계적으로 주장할 수 없다.

## STEP10 요약

### Takeaway

- DMD annotation(OpenLABEL)은 프레임 단위 눈 상태 GT 로 변환 가능하며, mosaic 프레임 번호와 오프셋 없이 1:1 대응한다.
- 사용 가능한 이진 GT 는 전체 프레임의 약 74%이고, 그 안에서 Closed:Open ≈ 1:6.4 로 심하게 불균형하다.
- 전이 프레임(opening / closing)은 정답이 애매해 평가에서 제외했다. 제외 비율이 약 26%라는 사실을 결과 해석에 반드시 함께 적는다.
- 안경 착용 subject 는 13명 중 3명뿐이어서 split 설계 시 이 3명의 배분이 핵심 제약이 된다.
- 이 노트북은 하품 라벨(`is_yawn` / `yawn_type`)도 CSV 에 함께 남기지만, 하품 파이프라인은 STEP20 · STEP21(팀원)에서 다룬다.